# SWOT PIXC: mapa interactivo por clases
Ejecuta la celda con un kernel de Jupyter que tenga `numpy`, `netCDF4`, `plotly`, `ipywidgets` y `anywidget`. Conserva la ruta original: ajusta `PIXC_FILE` si ejecutas en otra máquina. El zoom y el desplazamiento actualizan los puntos; un HTML exportado no ejecuta estos callbacks.


In [ ]:

# ============================================================
# SWOT PIXC - VISUALIZACIÓN 2D INTERACTIVA POR CLASIFICACIÓN
# Plotly | Sin raster | Sin máscara externa
# ============================================================

import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from ipywidgets import HTML, VBox
from netCDF4 import Dataset

# ============================================================
# OPTIMIZACIÓN DE MEMORIA
# ============================================================

# Máximo de píxeles representados por clase
MAX_PIXELES_POR_CLASE = 15000

# Semilla para que el muestreo sea reproducible
SEMILLA = 42

# Tamaño de puntos
TAMANO_PUNTO = 3

# Opacidad
OPACIDAD = 0.8

# ------------------------------------------------------------
# 1. ARCHIVO PIXC
# ------------------------------------------------------------

PIXC_FILE = (
    r'C:\Users\crist\Desktop\Doctorado\paper2\SWOT\swot_rivers\PIXC\SWOT_L2_HR_PIXC_001_451_087R_20230806T073922_20230806T073933_PGD0_01.nc'
)

# ------------------------------------------------------------
# 2. CLASES PIXC Y COLORES
# ------------------------------------------------------------

NOMBRES_CLASES = {
    1: "Land",
    2: "land_near_water",
    3: "water_near_land",
    4: "open_water",
    5: "dark_water",
    6: "low_coh_water_near_land",
    7: "open_low_coh_water"
}

COLORES_CLASES = {
    1: "#8B4513",  # Marrón
    2: "#DAA520",  # Dorado
    3: "#00BFFF",  # Celeste
    4: "#0066FF",  # Azul
    5: "#001B44",  # Azul muy oscuro
    6: "#00B894",  # Verde turquesa
    7: "#8E44AD"   # Morado
}

TAMANO_PUNTO = 3
OPACIDAD = 0.85

# ------------------------------------------------------------
# 3. LEER PIXC
# ------------------------------------------------------------

with Dataset(PIXC_FILE) as nc:

    pixc = nc.groups["pixel_cloud"]

    lat = np.ma.filled(
        pixc.variables["latitude"][:].astype(float),
        np.nan
    )

    lon = np.ma.filled(
        pixc.variables["longitude"][:].astype(float),
        np.nan
    )

    cls = np.ma.filled(
        pixc.variables["classification"][:].astype(float),
        np.nan
    )

# ------------------------------------------------------------
# 4. VALIDAR COORDENADAS Y CLASIFICACIONES
# ------------------------------------------------------------

mask_valid = (
    np.isfinite(lat) &
    np.isfinite(lon) &
    np.isfinite(cls) &
    np.isin(cls, list(NOMBRES_CLASES.keys()))
)

lat = lat[mask_valid]
lon = lon[mask_valid]
cls = cls[mask_valid].astype(np.uint8)
del mask_valid

print(f"Total de píxeles válidos: {len(cls):,}")

# ------------------------------------------------------------
# 5. RESUMEN DE CLASES
# ------------------------------------------------------------

print("\n===== DISTRIBUCIÓN DE CLASES PIXC =====")

for clase, nombre in NOMBRES_CLASES.items():

    cantidad = np.sum(cls == clase)

    porcentaje = (
        100 * cantidad / len(cls)
        if len(cls) > 0 else 0
    )

    print(
        f"{clase} | {nombre:28s} | "
        f"{cantidad:8,d} píxeles | {porcentaje:6.2f}%"
    )

# ------------------------------------------------------------
# 6. MAPA CON MUESTREO DINÁMICO SEGÚN ZOOM Y ÁREA VISIBLE
# Requiere kernel activo de Jupyter y anywidget.
# ------------------------------------------------------------

ZOOM_INICIAL = 5
# El presupuesto se duplica por nivel de zoom, hasta este límite.
MAX_PIXELES_POR_CLASE_ZOOM = 100000
ANCHO_MAPA = 1000
ALTO_MAPA = 700


def seleccionar_indices(lat, lon, cls, clase, centro, zoom):
    """Muestra determinista del área visible en proyección Web Mercator."""
    zoom = float(zoom)
    escala = 512 * 2.0 ** zoom
    centro_lat = np.clip(centro["lat"], -85.05112878, 85.05112878)
    centro_y = (1 - np.arcsinh(np.tan(np.radians(centro_lat))) / np.pi) / 2
    # Margen para incluir los bordes del mapa.
    mitad_y = ALTO_MAPA * 0.55 / escala
    lat_norte = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * (centro_y - mitad_y)))))
    lat_sur = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * (centro_y + mitad_y)))))
    mitad_lon = min(180, ANCHO_MAPA * 0.55 / escala * 360)
    indices = np.flatnonzero(
        (cls == clase) & (lat >= lat_sur) & (lat <= lat_norte)
        & (np.abs((lon - centro["lon"] + 180) % 360 - 180) <= mitad_lon)
    )
    presupuesto = min(
        MAX_PIXELES_POR_CLASE_ZOOM,
        max(1, int(MAX_PIXELES_POR_CLASE * 2.0 ** np.clip(zoom - ZOOM_INICIAL, -20, 20)))
    )
    if len(indices) > presupuesto:
        indices = np.random.default_rng(SEMILLA + clase).choice(
            indices, size=presupuesto, replace=False
        )
    return indices


if len(cls) == 0:
    print("No se encontraron píxeles PIXC válidos para graficar.")
else:
    # Cerrar el widget anterior al volver a ejecutar la celda.
    if "panel_mapa" in globals():
        panel_mapa.close()
        fig.close()
    fig = go.FigureWidget()
    clases_mapa = [clase for clase in NOMBRES_CLASES if np.any(cls == clase)]
    for clase in clases_mapa:
        nombre = NOMBRES_CLASES[clase]
        fig.add_trace(go.Scattermap(
            lat=[], lon=[], mode="markers",
            name=f"{clase} - {nombre}",
            marker=dict(size=TAMANO_PUNTO, color=COLORES_CLASES[clase], opacity=OPACIDAD),
            hovertemplate=(
                f"<b>{nombre}</b><br>Clase: {clase}<br>"
                "Longitud: %{lon:.6f}<br>Latitud: %{lat:.6f}<extra></extra>"
            )
        ))
    fig.update_layout(
        map=dict(style="open-street-map", zoom=ZOOM_INICIAL,
                 center=dict(lat=float(np.median(lat)), lon=float(np.median(lon)))),
        width=ANCHO_MAPA, height=ALTO_MAPA,
        margin=dict(l=0, r=0, t=0, b=0), uirevision="pixc-clases"
    )
    fig._config.update({
        "scrollZoom": True, "displayModeBar": True, "displaylogo": False,
        "toImageButtonOptions": {
            "format": "png", "filename": "SWOT_PIXC_classification",
            "height": 1000, "width": 1400, "scale": 2
        }
    })
    estado_mapa = HTML()

    def actualizar_mapa(layout, centro, zoom):
        centro = dict(lat=centro.lat, lon=centro.lon)
        total = 0
        with fig.batch_update():
            for traza, clase in zip(fig.data, clases_mapa):
                indices = seleccionar_indices(lat, lon, cls, clase, centro, zoom)
                # Conservar precisión de coordenadas para zoom alto.
                traza.lat = lat[indices]
                traza.lon = lon[indices]
                total += len(indices)
        estado_mapa.value = (
            f"Zoom: {zoom:.1f} | Píxeles representados: {total:,} / {len(cls):,}"
            " | Se muestran todos los visibles cuando caben en el presupuesto."
        )

    fig.layout.on_change(actualizar_mapa, "map.center", "map.zoom")
    actualizar_mapa(fig.layout, fig.layout.map.center, fig.layout.map.zoom)
    panel_mapa = VBox([estado_mapa, fig])
    display(panel_mapa)
